# NB1: Causal Engine (Zero LLM Calls)

**Purpose:** Prove the mechanical layer works before any LLM touches it.

**Tests:**
1. Scenario YAML loads into SQLite with all tables populated
2. Manual delta propagates through causal graph correctly
3. Decay/momentum evolves state plausibly over 5 turns with no actions
4. Clamping and rate limits enforced
5. Multi-turn actions progress correctly
6. All results match hand-calculated expected values

**Pass criteria:** All assertions pass. State trajectories are plausible when plotted.

In [ ]:
import sys
sys.path.insert(0, '..')

from wargame.scenario import load_scenario, init_db
from wargame.engine import (
    get_current_turn, get_variable, get_all_variables, get_variable_meta,
    apply_delta, apply_decay_and_momentum, propagate_causal_edges,
    apply_pending_effects, advance_multi_turn_actions, start_multi_turn_action,
    record_state_history, get_state_history, advance_turn, run_mechanical_phases,
)

SCENARIO_PATH = '../scenarios/us_iran_2026.yaml'

## Test 1: Scenario Loading

Load the YAML and verify all tables are populated.

In [ ]:
spec = load_scenario(SCENARIO_PATH)
print(f'Scenario: {spec.meta.name}')
print(f'Actors: {[a.id for a in spec.actors]}')
print(f'State variables: {len(spec.state_variables)}')
print(f'Causal edges: {len(spec.causal_edges)}')
print(f'Variable dynamics: {len(spec.variable_dynamics)}')
print(f'Multi-turn templates: {list(spec.multi_turn_actions.keys())}')

assert len(spec.actors) == 2
assert len(spec.state_variables) == 16
assert len(spec.causal_edges) == 14
print('\n✓ Scenario spec loaded and validated')

In [ ]:
conn = init_db(spec)

# Verify tables populated
actor_count = conn.execute('SELECT COUNT(*) FROM actors').fetchone()[0]
var_count = conn.execute('SELECT COUNT(*) FROM state_variables').fetchone()[0]
edge_count = conn.execute('SELECT COUNT(*) FROM causal_edges').fetchone()[0]
dyn_count = conn.execute('SELECT COUNT(*) FROM variable_dynamics').fetchone()[0]
est_count = conn.execute('SELECT COUNT(*) FROM state_estimates').fetchone()[0]
inst_count = conn.execute('SELECT COUNT(*) FROM instruments').fetchone()[0]
hist_count = conn.execute('SELECT COUNT(*) FROM state_history').fetchone()[0]

print(f'Actors: {actor_count}')
print(f'State variables: {var_count}')
print(f'Causal edges: {edge_count}')
print(f'Variable dynamics: {dyn_count}')
print(f'State estimates: {est_count} (should be actors × vars = {actor_count * var_count})')
print(f'Instruments: {inst_count}')
print(f'State history (turn 0): {hist_count}')

assert actor_count == 2
assert var_count == 16
assert edge_count == 14
assert dyn_count == 7
assert est_count == actor_count * var_count  # every actor has an estimate for every var
assert hist_count == var_count  # turn 0 snapshot
assert get_current_turn(conn) == 0

print('\n✓ Database initialized correctly')

In [ ]:
# Verify initial values
state = get_all_variables(conn)
print('Initial state:')
for var_id, value in sorted(state.items()):
    print(f'  {var_id}: {value}')

assert state['sv_nuclear_latency'] == 8.0
assert state['sv_sanctions_intensity'] == 0.7
assert state['sv_proxy_network_presence'] == 0.65
assert state['sv_military_tension'] == 0.4
print('\n✓ Initial state values correct')

## Test 2: Causal Propagation

Apply a manual delta to `sv_sanctions_intensity` and verify downstream effects.

Expected:
- `sv_sanctions_intensity` += 0.1
- Immediate (lag=0): `sv_perceived_us_coercion` += 0.1 * 0.25 = +0.025
- Lagged (lag=1): `sv_internal_legitimacy` will get 0.1 * -0.15 = -0.015 next turn
- Lagged (lag=2): `sv_elite_cohesion` will get 0.1 * -0.08 = -0.008 in 2 turns

In [ ]:
# Fresh DB for this test
conn2 = init_db(spec)
turn = advance_turn(conn2)  # Turn 1

# Record pre-state
pre_sanctions = get_variable(conn2, 'sv_sanctions_intensity')
pre_coercion = get_variable(conn2, 'sv_perceived_us_coercion')
pre_legitimacy = get_variable(conn2, 'sv_internal_legitimacy')
pre_cohesion = get_variable(conn2, 'sv_elite_cohesion')

print(f'Turn {turn}')
print(f'Pre-state:')
print(f'  sanctions_intensity: {pre_sanctions}')
print(f'  perceived_us_coercion: {pre_coercion}')
print(f'  internal_legitimacy: {pre_legitimacy}')
print(f'  elite_cohesion: {pre_cohesion}')

# Apply delta
actual = apply_delta(conn2, 'sv_sanctions_intensity', 0.1)
print(f'\nApplied delta to sv_sanctions_intensity: {actual}')

# Propagate
immediate = propagate_causal_edges(conn2, {'sv_sanctions_intensity': actual}, turn)
print(f'Immediate causal effects: {immediate}')

# Check immediate effect on perceived coercion
post_coercion = get_variable(conn2, 'sv_perceived_us_coercion')
expected_coercion = pre_coercion + (0.1 * 0.25)
print(f'\nPerceived US coercion: {pre_coercion} -> {post_coercion} (expected: {expected_coercion})')
assert abs(post_coercion - expected_coercion) < 1e-6, f'Expected {expected_coercion}, got {post_coercion}'

# Check that legitimacy hasn't changed yet (lag=1)
post_legitimacy = get_variable(conn2, 'sv_internal_legitimacy')
print(f'Internal legitimacy: {pre_legitimacy} -> {post_legitimacy} (should be unchanged)')
assert post_legitimacy == pre_legitimacy, 'Legitimacy should not have changed yet (lag=1)'

# Check pending effects
pending = conn2.execute('SELECT target_var, delta, applies_on_turn FROM pending_effects ORDER BY applies_on_turn').fetchall()
print(f'\nPending effects:')
for target, delta, applies in pending:
    print(f'  {target}: {delta:+.4f} on turn {applies}')

assert len(pending) == 2  # legitimacy (lag=1) and cohesion (lag=2)

# Now advance to turn 2 and apply pending effects
turn2 = advance_turn(conn2)
pending_deltas = apply_pending_effects(conn2, turn2)
print(f'\nTurn {turn2} - Applied pending effects: {pending_deltas}')

post_legitimacy2 = get_variable(conn2, 'sv_internal_legitimacy')
expected_legitimacy = pre_legitimacy + (0.1 * -0.15)
print(f'Internal legitimacy after lag: {post_legitimacy2} (expected: {expected_legitimacy})')
assert abs(post_legitimacy2 - expected_legitimacy) < 1e-6

# Elite cohesion should still be unchanged (lag=2, now only turn 2)
post_cohesion2 = get_variable(conn2, 'sv_elite_cohesion')
print(f'Elite cohesion: {post_cohesion2} (should still be {pre_cohesion})')
assert post_cohesion2 == pre_cohesion

# Advance to turn 3 for cohesion effect
turn3 = advance_turn(conn2)
pending_deltas3 = apply_pending_effects(conn2, turn3)
post_cohesion3 = get_variable(conn2, 'sv_elite_cohesion')
expected_cohesion = pre_cohesion + (0.1 * -0.08)
print(f'\nTurn {turn3} - Elite cohesion: {post_cohesion3} (expected: {expected_cohesion})')
assert abs(post_cohesion3 - expected_cohesion) < 1e-6

print('\n✓ Causal propagation (immediate + lagged) works correctly')

## Test 3: Decay and Momentum (5 turns, no actions)

Run 5 turns of pure decay/momentum with no player actions.

Expected behaviors:
- Flow vars (military_tension, diplomatic_engagement, proxy_activity) decay toward 0
- Structural vars (proxy_network_presence, force_posture) barely move
- Nuclear latency decreases by ~0.3/turn (momentum)
- Sanctions intensity decays slowly

In [ ]:
# Fresh DB
conn3 = init_db(spec)

print('Turn 0 (initial):')
initial = get_all_variables(conn3)
for var_id in sorted(initial):
    print(f'  {var_id}: {initial[var_id]:.3f}')

# Run 5 turns of mechanical phases only
for i in range(5):
    result = run_mechanical_phases(conn3)
    state = get_all_variables(conn3)
    print(f'\nTurn {result.turn_number}:')
    print(f'  Decay deltas: {result.decay_deltas}')
    print(f'  Multi-turn deltas: {result.multi_turn_deltas}')
    print(f'  Pending deltas: {result.pending_deltas}')
    for var_id in sorted(state):
        change = state[var_id] - initial[var_id]
        if abs(change) > 0.001:
            print(f'  {var_id}: {state[var_id]:.3f} (Δ{change:+.3f} from initial)')

In [ ]:
# Verify expected behaviors
final = get_all_variables(conn3)

# Nuclear latency should have decreased by ~1.5 (5 * -0.3)
nuclear_change = final['sv_nuclear_latency'] - initial['sv_nuclear_latency']
print(f'Nuclear latency change over 5 turns: {nuclear_change:.2f} (expected ~-1.5)')
assert -2.0 < nuclear_change < -1.0, f'Nuclear latency change {nuclear_change} seems wrong'

# Military tension should have decayed
tension_change = final['sv_military_tension'] - initial['sv_military_tension']
print(f'Military tension change: {tension_change:.3f} (should be negative)')
assert tension_change < 0, 'Military tension should decay'

# Structural vars should barely move
proxy_change = final['sv_proxy_network_presence'] - initial['sv_proxy_network_presence']
print(f'Proxy network presence change: {proxy_change:.4f} (should be small, decay=-0.01/turn)')
assert abs(proxy_change) < 0.1, 'Structural vars should move slowly'

# Sanctions should erode slightly
sanctions_change = final['sv_sanctions_intensity'] - initial['sv_sanctions_intensity']
print(f'Sanctions intensity change: {sanctions_change:.3f} (expected ~-0.1 over 5 turns)')
assert sanctions_change < 0, 'Sanctions should erode over time'

print('\n✓ Decay and momentum behave as expected')

## Test 4: Clamping and Rate Limits

In [ ]:
# Fresh DB
conn4 = init_db(spec)

# Test 4a: Range clamping — push sanctions_intensity past max (1.0)
pre = get_variable(conn4, 'sv_sanctions_intensity')  # 0.7
actual = apply_delta(conn4, 'sv_sanctions_intensity', 0.5)  # would make 1.2
post = get_variable(conn4, 'sv_sanctions_intensity')
print(f'Clamping test: {pre} + 0.5 requested -> {post} (actual delta: {actual})')
assert post == 1.0, f'Should be clamped to 1.0, got {post}'
assert abs(actual - 0.3) < 1e-6, f'Actual delta should be 0.3, got {actual}'

# Test 4b: Rate limit — structural variable limited to ±0.05
pre_struct = get_variable(conn4, 'sv_proxy_network_presence')  # 0.65
actual_struct = apply_delta(conn4, 'sv_proxy_network_presence', 0.20)  # should be capped at 0.05
post_struct = get_variable(conn4, 'sv_proxy_network_presence')
print(f'Rate limit test: {pre_struct} + 0.20 requested -> {post_struct} (actual delta: {actual_struct})')
assert abs(actual_struct - 0.05) < 1e-6, f'Structural rate limit should cap at 0.05, got {actual_struct}'

# Test 4c: Range clamping on the low end — push military tension below 0
pre_tens = get_variable(conn4, 'sv_military_tension')  # 0.4
actual_tens = apply_delta(conn4, 'sv_military_tension', -0.6)  # would make -0.2
post_tens = get_variable(conn4, 'sv_military_tension')
print(f'Lower clamp test: {pre_tens} + (-0.6) requested -> {post_tens} (actual delta: {actual_tens})')
assert post_tens == 0.0, f'Should be clamped to 0.0, got {post_tens}'

print('\n✓ Clamping and rate limits enforced correctly')

## Test 5: Multi-Turn Actions

In [ ]:
# Fresh DB
conn5 = init_db(spec)
advance_turn(conn5)  # Turn 1

# Start a sanctions ramp: effects [0.05, 0.10, 0.15] over 3 turns
pre_sanctions = get_variable(conn5, 'sv_sanctions_intensity')
print(f'Pre-sanctions: {pre_sanctions}')

active_id = start_multi_turn_action(
    conn5,
    template_name='sanctions_ramp',
    actor_id='actor_us',
    current_turn=1,
    target_var='sv_sanctions_intensity',
    effects_per_turn=[0.05, 0.10, 0.15],
    duration=3,
    resource_cost_per_turn=2,
    domain='finance',
)
print(f'Started multi-turn action (id={active_id})')

# Advance 3 turns, checking sanctions each time
expected_deltas = [0.05, 0.10, 0.15]
cumulative = 0.0
for i, expected in enumerate(expected_deltas):
    turn = advance_turn(conn5)
    mt_deltas = advance_multi_turn_actions(conn5, turn)
    sanctions = get_variable(conn5, 'sv_sanctions_intensity')
    cumulative += expected
    # Account for sanctions decay too (-0.02/turn)
    print(f'Turn {turn}: sanctions={sanctions:.3f}, mt_delta={mt_deltas}')
    assert 'sv_sanctions_intensity' in mt_deltas, f'Turn {turn}: expected sanctions delta'

# Verify action is marked completed
completed = conn5.execute('SELECT completed FROM active_actions WHERE active_id=?', (active_id,)).fetchone()[0]
assert completed == 1, 'Action should be marked completed'

# One more turn: no more multi-turn effect
turn = advance_turn(conn5)
mt_deltas = advance_multi_turn_actions(conn5, turn)
assert 'sv_sanctions_intensity' not in mt_deltas, 'Should be no more sanctions delta after completion'

print('\n✓ Multi-turn actions progress and complete correctly')

## Test 6: Full Mechanical Turn Sequence + Plotting

Run 10 turns of pure mechanical evolution and plot all state variable trajectories.

In [ ]:
import matplotlib.pyplot as plt

# Fresh DB with a sanctions ramp to make it interesting
conn6 = init_db(spec)

# Start a sanctions ramp on turn 1
advance_turn(conn6)
start_multi_turn_action(
    conn6, 'sanctions_ramp', 'actor_us', 1,
    'sv_sanctions_intensity', [0.05, 0.10, 0.15], 3, 2, 'finance'
)
record_state_history(conn6, 1)
conn6.commit()

# Run 10 turns
for i in range(10):
    result = run_mechanical_phases(conn6)

# Get full history and plot
history = get_state_history(conn6)

fig, axes = plt.subplots(4, 4, figsize=(20, 16))
fig.suptitle('State Variable Trajectories (10 turns, mechanical only + sanctions ramp)', fontsize=14)

for idx, (var_id, points) in enumerate(sorted(history.items())):
    ax = axes[idx // 4][idx % 4]
    turns = [p[0] for p in points]
    values = [p[1] for p in points]
    ax.plot(turns, values, 'b-o', markersize=3)
    ax.set_title(var_id.replace('sv_', ''), fontsize=9)
    ax.set_xlabel('Turn')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../notebooks/nb1_trajectories.png', dpi=100)
plt.show()
print('\nPlot saved to notebooks/nb1_trajectories.png')

In [ ]:
# Final state summary
final = get_all_variables(conn6)
initial = spec.initial_state

print('State after 10 mechanical turns + sanctions ramp:')
print(f'{"Variable":<35} {"Initial":>8} {"Final":>8} {"Change":>8}')
print('-' * 65)
for var_id in sorted(final):
    init_val = initial.get(var_id, 0)
    change = final[var_id] - init_val
    print(f'{var_id:<35} {init_val:>8.3f} {final[var_id]:>8.3f} {change:>+8.3f}')

## Summary

Check all pass criteria:
- [ ] Scenario YAML loads into SQLite with all tables populated
- [ ] Manual delta propagates through causal graph (immediate + lagged)
- [ ] 5 turns of decay: flow vars decay, structural barely move, nuclear latency decreases
- [ ] Clamping enforced (range limits + structural rate limits)
- [ ] Multi-turn actions progress and complete
- [ ] State trajectories plausible when plotted